# 04 · Validate — metal geometry, LigandMPNN-vs-ProteinMPNN, pool-size-vs-hit-rate, docking & MD

**Standard slot:** *validate (in silico).* **For Project 20 these are the benchmarks:** the
**metal-geometry preservation rate**, **LigandMPNN vs ProteinMPNN at the metal site**, and
**pool-size vs hit-rate** `[extension]`, plus the docking / caveated metal-site-MD figures (D3 pt2).

Needs `results/campaign.csv` (+ `results/ranked.csv` from notebook 03).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Metal-geometry preservation — the headline figure
Distribution of metal-ligand RMSD vs the 0.5 Å pass bar. The fraction left of the line is the
**preservation rate** — the metric that most distinguishes design tools and scaffolding methods.
(Numbers here are SYNTHETIC mock values; on Colab they come from real AF2 predictions with the Zn
built in.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")
cut = 0.5

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(camp["metal_ligand_rmsd"], bins=20)
ax.axvline(cut, color="k", ls="--", lw=1, label=f"pass < {cut} A")
ax.set_xlabel("metal-ligand RMSD vs target Zn-His3 (A)  [SYNTHETIC]")
ax.set_ylabel("designs"); ax.set_title("Metal-geometry preservation")
ax.legend(); plt.tight_layout()
plt.savefig("results/metal_geometry_hist.png", dpi=150); plt.show()

rate = 100 * (camp["metal_ligand_rmsd"] <= cut).mean()
print(f"overall metal-geometry preservation rate = {rate:.1f}%  [SYNTHETIC demo]")

## 2 · LigandMPNN vs ProteinMPNN at the metal site — the core benchmark
Does the **metal context** (LigandMPNN) hold the Zn-His₃ cage better than a **metal-blind** design
(ProteinMPNN, His fixed but no metal in context)? Compare the metal-geometry preservation rate by
`design_tool`. (Here the mock data gives LigandMPNN the edge by construction — a SYNTHETIC teaching
effect; on Colab the difference is whatever the real designs show.)

In [ ]:
by_tool = (camp.assign(pass_geom=camp["metal_ligand_rmsd"] <= 0.5)
               .groupby("design_tool")
               .agg(n=("design_id", "size"),
                    geom_pass_rate=("pass_geom", "mean"),
                    mean_metal_rmsd=("metal_ligand_rmsd", "mean"),
                    mean_plddt_cat=("plddt_catalytic", "mean"))
               .reset_index())
by_tool["geom_pass_rate"] = (100 * by_tool["geom_pass_rate"]).round(1)
print("LigandMPNN (metal-aware) vs ProteinMPNN (metal-blind) at the Zn-His3 site [SYNTHETIC]:")
print(by_tool.to_string(index=False))

fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.bar(by_tool["design_tool"], by_tool["geom_pass_rate"])
ax.set_ylabel("metal-geometry pass rate (%)  [SYNTHETIC]")
ax.set_title("Metal context helps hold the cage")
for i, v in enumerate(by_tool["geom_pass_rate"]):
    ax.text(i, v, f"{v}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig("results/ligandmpnn_vs_proteinmpnn.png", dpi=150); plt.show()
print("On Colab: run BOTH tools on the SAME backbones (His fixed); the only difference is the Zn context.")

## 3 · Pool-size vs hit-rate — does scaling toward ~10k pay off? `[extension]`
GRACE generated a **large pool (~10k)**. Subsample the pool at increasing sizes and plot the cumulative
count of designs passing the metal-geometry bar. A roughly linear count (flat *rate*) means hits scale
with pool size — the rationale for a large campaign. (SYNTHETIC mock pool here; the *shape* is the point.)

In [ ]:
rng = np.random.default_rng(0)
lig = camp[camp["design_tool"] == "ligandmpnn"].copy()
passes = (lig["metal_ligand_rmsd"] <= 0.5).to_numpy()
order = rng.permutation(len(passes))
sizes = np.linspace(1, len(passes), min(20, len(passes))).astype(int)
cum_hits = [int(passes[order[:k]].sum()) for k in sizes]

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.plot(sizes, cum_hits, marker="o")
ax.set_xlabel("pool size (designs screened)  [SYNTHETIC mock pool]")
ax.set_ylabel("cumulative metal-geometry hits")
ax.set_title("Pool-size vs hit count (extrapolate toward ~10k)")
plt.tight_layout(); plt.savefig("results/pool_size_vs_hits.png", dpi=150); plt.show()
rate = 100 * passes.mean()
print(f"hit RATE is ~constant at {rate:.1f}% [SYNTHETIC] -> hit COUNT grows with pool size; "
      "this is why GRACE used ~10k. On Colab, plot your real pool.")

## 4 · Substrate fit + metal-site MD (top candidates) — with the metal-FF caveat
Docking (AutoDock Vina) checks the substrate (CO₂ / the pNPA proxy) **reaches and is oriented toward
the Zn-OH** — not affinity, not activity. Short MD (OpenMM) checks the metal site doesn't drift — **but
classical fixed-charge force fields model a coordinated metal poorly**, so this is a *weak proxy*, not
ground truth. Plot the two for the ranked survivors as orthogonal (caveated) evidence.

In [ ]:
try:
    ranked = pd.read_csv("results/ranked.csv")
    ranked = ranked.merge(camp[["design_id", "vina_score"]], on="design_id", how="left")
except FileNotFoundError:
    ranked = camp.copy()

top = ranked.head(min(20, len(ranked)))
fig, ax = plt.subplots(figsize=(5.2, 3.4))
sc = ax.scatter(top["vina_score"], top["md_rmsd"],
                c=top["catalytic_geom_rmsd"], cmap="viridis")
ax.set_xlabel("Vina substrate-fit score (more negative = better fit)  [SYNTHETIC]")
ax.set_ylabel("metal-site MD RMSD (A)  [SYNTHETIC; metal-FF caveat]")
ax.set_title("Top candidates: substrate fit vs metal-site stability")
fig.colorbar(sc, label="metal-ligand RMSD (A)")
plt.tight_layout(); plt.savefig("results/docking_md.png", dpi=150); plt.show()
print("Lower-left + dark points (good fit, stable, good cage) are best [SYNTHETIC].")
print("CAVEAT: classical MD cannot model the Zn centre well — treat 'stable' as necessary, not sufficient.")

## 5 · Honest hit-rate accounting
Report N(pass all layers) / N(generated), and remind the reader of the field reality: even a good
preservation rate is **not** an activity rate, **and not even a metal-incorporation rate**. Geometry ≠
incorporation ≠ catalysis; ICP and a kinetic assay are required.

In [ ]:
n_total = len(camp)
try:
    ranked = pd.read_csv("results/ranked.csv")
    n_hits = int((ranked["layers_passed"] >= 3).sum())
except Exception:
    n_hits = int((camp["metal_ligand_rmsd"] <= 0.5).sum())
print("Hit-rate accounting [SYNTHETIC demo]:")
print(f"  generated             : {n_total}")
print(f"  pass all filter layers: {n_hits}  ({100*n_hits/max(n_total,1):.1f}%)")
print("\nREALITY CHECK: de novo metalloenzyme activity rates are low (GRACE needed ~10k + screening).")
print("In-silico metal geometry does NOT guarantee activity, NOR that Zn binds. ICP + an assay decide.")

## D3 (part 2) checklist
- [ ] Metal-geometry preservation histogram (`results/metal_geometry_hist.png`) + rate.
- [ ] **LigandMPNN-vs-ProteinMPNN** metal-site benchmark (`results/ligandmpnn_vs_proteinmpnn.png`).
- [ ] **Pool-size-vs-hit-rate** figure (`results/pool_size_vs_hits.png`) `[extension]`.
- [ ] Docking + (caveated) metal-site-MD figure on the ranked top set.
- [ ] Honest hit-rate accounting with the "geometry ≠ incorporation ≠ activity" + metal-FF caveats stated.

**Next:** `05_validation_plan.ipynb` — the activity + metal-incorporation assay plan + controls + Co-substitution stretch.